# Lezione 3 - Pratica: Visione artificiale II, architetture avanzate

In questa pratica applichi quanto visto nella Lezione 3 sulle reti neurali convoluzionali (CNN) avanzate: le connessioni residue, che rendono possibile addestrare reti molto profonde, e il transfer learning, che riutilizza un modello preaddestrato per risolvere un nuovo compito con pochi dati.

## Cosa farai in questa pratica

- Costruisci da zero un blocco residuo in PyTorch e verifichi che la connessione di scavalcamento (*skip*) funzioni come previsto.
- Adatti una ResNet-18 preaddestrata a un problema di classificazione binaria con poche immagini, tramite transfer learning.

## Prerequisito

Prima di iniziare, conferma di aver seguito la Lezione 3, "Visione artificiale II: architetture avanzate". Devi capire cos'è una connessione residua e in cosa consiste il transfer learning per completare i due esercizi.

Esegui questo notebook in Google Colab. Non installare nulla di aggiuntivo: PyTorch e torchvision sono già preinstallati nell'ambiente.

## Esercizio 1 — Costruisci un blocco residuo da zero

Il tuo team vuole capire, riga per riga, come funziona il pezzo centrale di ResNet (Residual Network, "rete residua") prima di usarla come scatola nera in un progetto. Ti chiedono di implementare la classe `BloqueResidual`, il componente di base che si ripete decine di volte all'interno di una rete residua.

Un blocco residuo applica due convoluzioni 3x3 con normalizzazione per lotti (batch normalization) e, prima dell'attivazione finale, somma l'ingresso originale del blocco all'uscita di quelle convoluzioni. Questa somma è la connessione residua: invece di apprendere l'intera trasformazione H(x), il blocco apprende solo il residuo F(x) = H(x) − x, e ricostruisce l'uscita come y = F(x) + x.

Quando il blocco cambia il numero di canali o la risoluzione spaziale (per esempio con `stride=2`), l'ingresso x non ha la stessa forma di F(x), quindi non puoi sommarli direttamente. Per questo lo scheletro include già una scorciatoia (`self.atajo`) che applica una convoluzione 1x1 e la normalizzazione per lotti solo quando serve adattare quella forma.

**Suggerimento:** il metodo `__init__` è già completo. Il tuo compito è terminare il metodo `forward`: manca una sola riga, quella che somma `salida` e `identidad` prima di restituire il risultato con l'attivazione applicata.

In [ ]:
import torch
import torch.nn as nn


class BloqueResidual(nn.Module):
    """Bloque residual básico: dos convoluciones 3x3 y un atajo (shortcut)
    que suma la entrada a la salida antes de la última activación."""

    def __init__(self, canales_entrada, canales_salida, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(canales_entrada, canales_salida, kernel_size=3,
                                stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(canales_salida)
        self.conv2 = nn.Conv2d(canales_salida, canales_salida, kernel_size=3,
                                stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(canales_salida)
        self.relu = nn.ReLU(inplace=True)

        # Si el bloque cambia la resolución o el número de canales, el atajo
        # necesita una proyección 1x1 para poder sumarse a F(x) más adelante
        self.atajo = nn.Sequential()
        if stride != 1 or canales_entrada != canales_salida:
            self.atajo = nn.Sequential(
                nn.Conv2d(canales_entrada, canales_salida, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(canales_salida),
            )

    def forward(self, x):
        identidad = self.atajo(x)                # esta es la vía directa: F(x) + x
        salida = self.relu(self.bn1(self.conv1(x)))
        salida = self.bn2(self.conv2(salida))

        # TODO: suma la conexión residual antes de la activación final.
        # Combina "salida" e "identidad" y guarda el resultado en "salida"
        salida = ...

        return self.relu(salida)

In [ ]:
# Prueba rápida: comprueba que el bloque produce la forma de salida esperada
entrada_prueba = torch.randn(2, 3, 32, 32)  # lote de 2 imágenes, 3 canales, 32x32 píxeles
bloque = BloqueResidual(canales_entrada=3, canales_salida=16, stride=2)
print(bloque(entrada_prueba).shape)  # forma esperada: torch.Size([2, 16, 16, 16])

<details>
<summary>💡 Vedi soluzione</summary>

```python
import torch
import torch.nn as nn


class BloqueResidual(nn.Module):
    """Blocco residuo di base: due convoluzioni 3x3 e una scorciatoia (shortcut)
    che somma l'ingresso all'uscita prima dell'ultima attivazione."""

    def __init__(self, canales_entrada, canales_salida, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(canales_entrada, canales_salida, kernel_size=3,
                                stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(canales_salida)
        self.conv2 = nn.Conv2d(canales_salida, canales_salida, kernel_size=3,
                                stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(canales_salida)
        self.relu = nn.ReLU(inplace=True)

        self.atajo = nn.Sequential()
        if stride != 1 or canales_entrada != canales_salida:
            self.atajo = nn.Sequential(
                nn.Conv2d(canales_entrada, canales_salida, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(canales_salida),
            )

    def forward(self, x):
        identidad = self.atajo(x)
        salida = self.relu(self.bn1(self.conv1(x)))
        salida = self.bn2(self.conv2(salida))

        # La connessione residua: sommiamo l'ingresso (adattato dalla scorciatoia
        # se necessario) a ciò che hanno calcolato le due convoluzioni, e solo
        # allora applichiamo l'ultima attivazione
        salida = salida + identidad

        return self.relu(salida)


entrada_prueba = torch.randn(2, 3, 32, 32)
bloque = BloqueResidual(canales_entrada=3, canales_salida=16, stride=2)
print(bloque(entrada_prueba).shape)  # torch.Size([2, 16, 16, 16])
```

Se commenti la riga `salida = salida + identidad`, il blocco continua a eseguirsi senza errori, ma smette di essere residuo: ogni blocco dovrebbe ricostruire l'intera trasformazione da zero, invece di apprendere solo la differenza rispetto al lasciar passare l'ingresso. Con reti di decine di strati, questa differenza è ciò che decide se l'addestramento converge o si blocca.
</details>

## Esercizio 2 — Classificatore a due categorie con transfer learning

Un ente ti chiede un classificatore che distingua fotografie di due categorie di prodotto, ma dispone solo di una manciata di immagini etichettate per categoria: non bastano per addestrare una rete convoluzionale da zero. Applichi il transfer learning: parti da una ResNet-18 già addestrata su ImageNet, conservi tutto ciò che ha appreso su bordi, texture e forme, e addestri solo uno strato nuovo adattato al tuo problema.

Per questa pratica simuli questo scenario con il dataset CIFAR-10: tieni solo due delle sue dieci classi (gatto e cane) e un sottoinsieme ridotto di immagini per classe, per riprodurre la situazione di "pochi dati".

Lo scheletro del caricamento dei dati e il ciclo di addestramento sono già completi. Ti manca la parte centrale del transfer learning, nella cella che definisce il modello:

1. Congela tutti i parametri della ResNet-18 preaddestrata, in modo che l'addestramento non li modifichi.
2. Sostituisci l'ultimo strato (`modelo.fc`) con uno strato lineare nuovo, adattato al numero di classi del tuo problema. Questo strato nuovo sì che si addestra: creandolo da zero, i suoi parametri nascono con `requires_grad = True`.

**Suggerimento:** per congelare un parametro in PyTorch, imposta il suo attributo `.requires_grad` a `False`. Scorri `modelo.parameters()` con un ciclo. Lo strato originale `modelo.fc` è un `nn.Linear` che riceve `modelo.fc.in_features` caratteristiche in ingresso: usa lo stesso numero quando crei lo strato di sostituzione.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, Subset, Dataset

dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dos categorías de CIFAR-10: 3 = gato, 5 = perro
CLASES_ELEGIDAS = [3, 5]
NUM_CLASES = len(CLASES_ELEGIDAS)
IMAGENES_POR_CLASE = 200  # subconjunto pequeño, para simular un escenario de pocos datos

preprocesar = transforms.Compose([
    transforms.Resize(224),  # ResNet-18 espera imágenes de 224x224 píxeles
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],   # medias de ImageNet
                          std=[0.229, 0.224, 0.225]),
])

dataset_completo = torchvision.datasets.CIFAR10(
    root="./datos", train=True, download=True, transform=preprocesar
)

# Nos quedamos solo con las imágenes de las dos clases elegidas
etiquetas = torch.tensor(dataset_completo.targets)
indices_filtrados = []
for clase in CLASES_ELEGIDAS:
    indices_clase = (etiquetas == clase).nonzero(as_tuple=True)[0].tolist()
    indices_filtrados.extend(indices_clase[:IMAGENES_POR_CLASE])

subconjunto = Subset(dataset_completo, indices_filtrados)


class DatasetDosClases(Dataset):
    """Envuelve el subconjunto y remapea las etiquetas originales (3, 5)
    a (0, 1), que es lo que espera una capa lineal de dos salidas."""

    def __init__(self, subconjunto, clases_elegidas):
        self.subconjunto = subconjunto
        self.clases_elegidas = clases_elegidas

    def __len__(self):
        return len(self.subconjunto)

    def __getitem__(self, idx):
        imagen, etiqueta_original = self.subconjunto[idx]
        etiqueta_nueva = self.clases_elegidas.index(etiqueta_original)
        return imagen, etiqueta_nueva


dataset_entrenamiento = DatasetDosClases(subconjunto, CLASES_ELEGIDAS)
cargador_entrenamiento = DataLoader(dataset_entrenamiento, batch_size=16, shuffle=True)

print(f"Imágenes de entrenamiento: {len(dataset_entrenamiento)}")

In [ ]:
modelo = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)

# TODO: congela todos los parámetros de la ResNet-18 preentrenada.
# Recorre modelo.parameters() con un bucle y pon requires_grad = False
# en cada parámetro, para que el entrenamiento no los modifique
...

# La capa modelo.fc original es un nn.Linear que mapea a las 1000 clases de ImageNet.
# TODO: sustitúyela por un nn.Linear nuevo, con modelo.fc.in_features características
# de entrada y NUM_CLASES de salida. Al crearse desde cero, sus parámetros nacen
# con requires_grad = True, así que serán los únicos que se entrenen
modelo.fc = ...

modelo = modelo.to(dispositivo)

In [ ]:
criterio = nn.CrossEntropyLoss()
# Solo optimizamos los parámetros de la capa nueva: el resto quedó congelado
optimizador = optim.Adam(modelo.fc.parameters(), lr=1e-3)

modelo.train()
NUM_EPOCAS = 3
for epoca in range(NUM_EPOCAS):
    perdida_acumulada = 0.0
    for imagenes, etiquetas_lote in cargador_entrenamiento:
        imagenes = imagenes.to(dispositivo)
        etiquetas_lote = etiquetas_lote.to(dispositivo)

        optimizador.zero_grad()
        salidas = modelo(imagenes)
        perdida = criterio(salidas, etiquetas_lote)
        perdida.backward()
        optimizador.step()

        perdida_acumulada += perdida.item()

    perdida_media = perdida_acumulada / len(cargador_entrenamiento)
    print(f"Época {epoca + 1}/{NUM_EPOCAS} - pérdida media: {perdida_media:.4f}")

<details>
<summary>💡 Vedi soluzione</summary>

```python
modelo = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)

# Congeliamo tutti i parametri preaddestrati: non vogliamo che l'addestramento
# con pochi dati li modifichi e rovini ciò che è già stato appreso
for parametro in modelo.parameters():
    parametro.requires_grad = False

# Sostituiamo l'ultimo strato con uno nuovo, adattato alle nostre due classi.
# I suoi parametri nascono con requires_grad = True, quindi saranno gli unici
# che l'ottimizzatore aggiorna durante l'addestramento
modelo.fc = nn.Linear(modelo.fc.in_features, NUM_CLASES)

modelo = modelo.to(dispositivo)
```

Osserva il risultato: anche se la ResNet-18 completa ha milioni di parametri, addestri solo quelli dello strato `fc` nuovo, qualche migliaio. Per questo il transfer learning funziona con così poche immagini: il modello non deve reimparare a riconoscere bordi e texture, deve solo combinare le caratteristiche che già estrae per separare le tue due categorie.
</details>

## Per concludere

Hai appena messo in pratica due idee centrali della visione artificiale moderna: la connessione residua, che permette di addestrare reti molto profonde, e il transfer learning, che fa risparmiare dati e calcolo riutilizzando ciò che un altro modello ha già appreso. Sono gli stessi elementi che sostengono rilevatori come Faster R-CNN o RetinaNet e reti di segmentazione come U-Net, architetture che vedrai più avanti nel corso.

Se qualcosa non ha funzionato al primo tentativo, controlla le forme dei tensori in ogni passo: la maggior parte degli errori in questi esercizi deriva da una discrepanza tra canali o tra dimensioni.

**Apri il notebook in Google Colab, completa i due esercizi senza guardare la soluzione prima di averci provato e condividi i tuoi dubbi nel forum del corso.**